In [2]:
# 12 — Cell 1: BanglaBERT GroupKFold(5) setup (same settings as 09)
import os, numpy as np, pandas as pd, torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

P = "/mnt/g/banglafake-detection/data/processed/v2"
R = "/mnt/g/banglafake-detection/reports/v2"
OUT = f"{R}/bb_groupkfold"; os.makedirs(OUT, exist_ok=True)
MODEL, MAXLEN, SEED = "csebuetnlp/banglabert", 256, 0

try:
    from normalizer import normalize as bn_normalize
except ImportError:
    bn_normalize = lambda s: s
    print("WARNING: normalizer not found, running without it (09-তে থাকলে এখানেও থাকা উচিত)")

CORP = {n: pd.read_csv(f"{P}/{n}.csv") for n in ("c2020", "c2025")}
for df in CORP.values():
    df["text"] = df["text"].fillna("").str[:2000].map(bn_normalize)
    df["domain"] = df["domain"].fillna("unk")
    df["label"] = df["label"].astype(int)

tok = AutoTokenizer.from_pretrained(MODEL, use_fast=False)

class TxtDS(torch.utils.data.Dataset):
    def __init__(self, df):
        self.enc = tok(df["text"].tolist(), truncation=True, max_length=MAXLEN, padding=False)
        self.labels = df["label"].tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights=None, **kw):
        super().__init__(*a, **kw); self.cw = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = nn.functional.cross_entropy(out.logits, labels, weight=self.cw.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def fit_scores(tr, te):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
    vc = tr["label"].value_counts()
    cw = torch.tensor([len(tr)/(2*vc.get(0,1)), len(tr)/(2*vc.get(1,1))], dtype=torch.float)
    args = TrainingArguments(
        output_dir="/tmp/bb_gk", per_device_train_batch_size=16, per_device_eval_batch_size=64,
        num_train_epochs=3, learning_rate=2e-5, weight_decay=0.01, logging_steps=200,
        save_strategy="no", report_to=[], seed=SEED, fp16=torch.cuda.is_available())
    trainer = WeightedTrainer(model=model, args=args, train_dataset=TxtDS(tr),
                              data_collator=lambda f: tok.pad(f, return_tensors="pt"),
                              class_weights=cw)
    trainer.train()
    logits = trainer.predict(TxtDS(te)).predictions
    del trainer, model; torch.cuda.empty_cache()
    return logits[:, 1] - logits[:, 0]

def run_groupkfold(name):
    df = CORP[name]
    for fold, (tr_i, te_i) in enumerate(GroupKFold(n_splits=5).split(df, groups=df["domain"])):
        f_s, f_i = f"{OUT}/{name}_fold{fold}_scores.npy", f"{OUT}/{name}_fold{fold}_idx.npy"
        if os.path.exists(f_s):
            print(f"[{name}] fold {fold} আগেই শেষ, skip"); continue
        print(f"[{name}] fold {fold}: train={len(tr_i)} test={len(te_i)}")
        s = fit_scores(df.iloc[tr_i], df.iloc[te_i])
        np.save(f_i, te_i); np.save(f_s, s)

def pooled(name):
    df = CORP[name]; ys, ss = [], []
    for fold in range(5):
        te_i = np.load(f"{OUT}/{name}_fold{fold}_idx.npy"); s = np.load(f"{OUT}/{name}_fold{fold}_scores.npy")
        ys.append(df.iloc[te_i]["label"].values); ss.append(s)
    y, s = np.concatenate(ys), np.concatenate(ss); p = (s > 0).astype(int)
    r = dict(corpus=name, model="BanglaBERT", n=len(y),
             bacc=round(balanced_accuracy_score(y, p), 3),
             recall0=round(((p==0)&(y==0)).sum()/(y==0).sum(), 3),
             recall1=round(((p==1)&(y==1)).sum()/(y==1).sum(), 3),
             auc=round(roc_auc_score(y, s), 3))
    pd.DataFrame([r]).to_csv(f"{R}/groupkfold_banglabert_{name}.csv", index=False)
    return r
print("setup ok | GPU:", torch.cuda.is_available())

setup ok | GPU: True


In [2]:
run_groupkfold("c2025")
print(pooled("c2025"))

[c2025] fold 0: train=2562 test=1417


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.233100
400,0.110900


[c2025] fold 1: train=3347 test=632


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Step,Training Loss
200,0.189800
400,0.042800
600,0.031500


[c2025] fold 4: train=3336 test=643


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.185700
400,0.043400
600,0.031200


{'corpus': 'c2025', 'model': 'BanglaBERT', 'n': 3979, 'bacc': 0.711, 'recall0': np.float64(0.435), 'recall1': np.float64(0.987), 'auc': 0.979}


In [3]:
run_groupkfold("c2020")
print(pooled("c2020"))

[c2020] fold 0: train=39869 test=9952


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.439000
400,0.255700
600,0.241400
800,0.298100
1000,0.223800
1200,0.175100
1400,0.255200
1600,0.142700
1800,0.285200
2000,0.211900


[c2020] fold 1: train=39869 test=9952


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Step,Training Loss
200,0.591500
400,0.473900
600,0.394200
800,0.344800
1000,0.310300
1200,0.329700
1400,0.309000
1600,0.314800
1800,0.237800
2000,0.170600


[c2020] fold 4: train=39808 test=10013


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.529200
400,0.370600
600,0.270000
800,0.283400
1000,0.242300
1200,0.168900
1400,0.215100
1600,0.288600
1800,0.179800
2000,0.144800


{'corpus': 'c2020', 'model': 'BanglaBERT', 'n': 49821, 'bacc': 0.794, 'recall0': np.float64(0.591), 'recall1': np.float64(0.998), 'auc': 0.944}
